In [1]:
import random

from apriori.ico.core.source import IcoSource
from apriori.ico.utils.data.batcher import IcoBatcher
from examples.visual.cifar.dataset import CifarInMemoryDataset

batch_size = 64
train_split_ratio = 0.02

dataset = CifarInMemoryDataset(share_memory=False)
dataset_size = len(dataset)
all_indices = list(range(dataset_size))
random.shuffle(all_indices)

train_split = int(dataset_size * train_split_ratio)
train_indices = all_indices[:train_split]

train_source = IcoSource[int](lambda: iter(train_indices), name="CIFAR10 train indices")
train_source.describe()



────────────────────────────────────────────── CIFAR10 train indices ──────────────────────────────────────────────

CIFAR10 train indices (source)  () → Iterator[int]

In [2]:
from collections.abc import Iterator

from apriori.ico.core.operator import IcoOperator
from examples.visual.cifar.dataset import CifarItem


def shuffle_indices(indices: Iterator[int]) -> Iterator[int]:
    indices_list = list(indices)
    random.shuffle(indices_list)
    yield from indices_list


def fetch_item(index: int) -> CifarItem:
    return dataset[index]


batcher = IcoBatcher[CifarItem](batch_size=batch_size)

train_input_flow = train_source | IcoOperator(shuffle_indices) | IcoOperator(fetch_item).iterate() | batcher
train_input_flow.name = "Train Input Flow"
train_input_flow.describe()

──────────────────────────────────────────────── Train Input Flow ─────────────────────────────────────────────────

Train Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
├── CIFAR10 train indices | shuffle_indices | iterate(fetch_item) (chain)  () → Iterator[CifarItem]
│   ├── CIFAR10 train indices | shuffle_indices (chain)  () → Iterator[int]
│   │   ├── CIFAR10 train indices (source)  () → Iterator[int]
│   │   └── shuffle_indices (operator)  Iterator[int] → Iterator[int]
│   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
│       └── fetch_item (operator)  int → CifarItem
└── batcher(64) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]

In [3]:
from examples.visual.cifar.data_flow import (
    create_augmentation_flow,
)

augment_flow = create_augmentation_flow(share_tensors=False)

train_data_flow = train_input_flow | augment_flow.iterate()
train_data_flow.name = "Single Process Data Flow with Augmentation"
train_data_flow.describe()

─────────────────────────────────── Single Process Data Flow with Augmentation ────────────────────────────────────

Single Process Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
├── Train Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
│   ├── CIFAR10 train indices | shuffle_indices | iterate(fetch_item) (chain)  () → Iterator[CifarItem]
│   │   ├── CIFAR10 train indices | shuffle_indices (chain)  () → Iterator[int]
│   │   │   ├── CIFAR10 train indices (source)  () → Iterator[int]
│   │   │   └── shuffle_indices (operator)  Iterator[int] → Iterator[int]
│   │   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
│   │       └── fetch_item (operator)  int → CifarItem
│   └── batcher(64) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
└── iterate(Full Augmentation flow) (iterator)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
    └── Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
        ├── iterate(Item augmentation flow) (iterator)  Iterator[CifarItem] → Iterator[CifarItem]
        │   └── Item augmentation flow (pipeline)  CifarItem → CifarItem
        │       ├── HorizontalFlip (operator)  CifarItem → CifarItem
        │       ├── VerticalFlip (operator)  CifarItem → CifarItem
        │       ├── AdjustBrightness (operator)  CifarItem → CifarItem
        │       └── AdjustContrast (operator)  CifarItem → CifarItem
        └── collate (operator)  Iterator[CifarItem] → CifarBatch

In [4]:
for batch in train_data_flow(None):
    print(batch)
    break

In [5]:
import torch.nn as nn
from torchvision.models import resnet18


def create_cifar10_resnet18(num_classes: int = 10):
    model = resnet18()
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, num_classes)
    return model

model = create_cifar10_resnet18()
outputs = model(batch.images)
print(f"Outputs shape: {outputs.shape}")

Outputs shape: torch.Size([64, 10])


In [6]:
from dataclasses import dataclass

import torch

from apriori.ico.core.context_pipeline import IcoContextPipeline
from apriori.ico.core.meta.describer import describe
from examples.visual.cifar.data_flow import CifarBatch


@dataclass(slots=True)
class CifarContext:
    model: nn.Module
    optimizer: torch.optim.Optimizer
    loss_fn: torch.nn.Module = torch.nn.CrossEntropyLoss()
    iter_num: int = 0
    total_loss: float = 0.0


def train_step(batch: CifarBatch, context: CifarContext) -> CifarContext:
    context.optimizer.zero_grad()

    outputs = context.model(batch.images)
    loss = context.loss_fn(outputs, batch.labels)
    loss.backward()
    context.optimizer.step()

    context.total_loss = loss.item()
    context.iter_num += 1
    return context


def logging_step(context: CifarContext) -> CifarContext:
    if context.iter_num % 10 == 0:
        print(f"Iteration {context.iter_num}, Loss: {context.total_loss:.4f}")
    return context

def save_checkpoint_step(context: CifarContext) -> CifarContext:
    if context.iter_num % 100 == 0:
        print(f"Checkpointing model at iteration {context.iter_num}")
    return context



train_pipeline = IcoContextPipeline(
    train_step,
    logging_step,
    save_checkpoint_step
)
train_pipeline.name = "Training Pipeline"
describe(train_pipeline)


──────────────────────────────────────────────── Training Pipeline ────────────────────────────────────────────────

Training Pipeline (context_pipeline)  CifarBatch, CifarContext → CifarContext
├── train_step (context_operator)  CifarBatch, CifarContext → CifarContext
├── logging_step (operator)  CifarContext → CifarContext
└── save_checkpoint_step (operator)  CifarContext → CifarContext

In [7]:
from apriori.ico.core.context_stream import IcoEpoch

train_epoch = IcoEpoch(
    source=train_data_flow,
    context_operator=train_pipeline,
)
train_epoch.name = "Training Epoch"
train_epoch.describe()

───────────────────────────────────────────────── Training Epoch ──────────────────────────────────────────────────

Training Epoch (epoch)  Iterator[CifarBatch], CifarContext → CifarContext
├── Single Process Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
│   ├── Train Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
│   │   ├── CIFAR10 train indices | shuffle_indices | iterate(fetch_item) (chain)  () → Iterator[CifarItem]
│   │   │   ├── CIFAR10 train indices | shuffle_indices (chain)  () → Iterator[int]
│   │   │   │   ├── CIFAR10 train indices (source)  () → Iterator[int]
│   │   │   │   └── shuffle_indices (operator)  Iterator[int] → Iterator[int]
│   │   │   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
│   │   │       └── fetch_item (operator)  int → CifarItem
│   │   └── batcher(64) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
│   └── iterate(Full Augmentation flow) (iterator)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
│       └── Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
│           ├── iterate(Item augmentation flow) (iterator)  Iterator[CifarItem] → Iterator[CifarItem]
│           │   └── Item augmentation flow (pipeline)  CifarItem → CifarItem
│           │       ├── HorizontalFlip (operator)  CifarItem → CifarItem
│           │       ├── VerticalFlip (operator)  CifarItem → CifarItem
│           │       ├── AdjustBrightness (operator)  CifarItem → CifarItem
│           │       └── AdjustContrast (operator)  CifarItem → CifarItem
│           └── collate (operator)  Iterator[CifarItem] → CifarBatch
└── Training Pipeline (context_pipeline)  CifarBatch, CifarContext → CifarContext
    ├── train_step (context_operator)  CifarBatch, CifarContext → CifarContext
    ├── logging_step (operator)  CifarContext → CifarContext
    └── save_checkpoint_step (operator)  CifarContext → CifarContext

In [8]:
def start_train(context: CifarContext) -> CifarContext:
    context.model.train()
    return context

train_flow = IcoOperator(start_train) | train_epoch
train_flow.name = "Train Flow"
train_flow.describe()



─────────────────────────────────────────────────── Train Flow ────────────────────────────────────────────────────

Train Flow (chain)  CifarContext → CifarContext
├── start_train (operator)  CifarContext → CifarContext
└── Training Epoch (epoch)  Iterator[CifarBatch], CifarContext → CifarContext
    ├── Single Process Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
    │   ├── Train Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
    │   │   ├── CIFAR10 train indices | shuffle_indices | iterate(fetch_item) (chain)  () → Iterator[CifarItem]
    │   │   │   ├── CIFAR10 train indices | shuffle_indices (chain)  () → Iterator[int]
    │   │   │   │   ├── CIFAR10 train indices (source)  () → Iterator[int]
    │   │   │   │   └── shuffle_indices (operator)  Iterator[int] → Iterator[int]
    │   │   │   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
    │   │   │       └── fetch_item (operator)  int → CifarItem
    │   │   └── batcher(64) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
    │   └── iterate(Full Augmentation flow) (iterator)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
    │       └── Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
    │           ├── iterate(Item augmentation flow) (iterator)  Iterator[CifarItem] → Iterator[CifarItem]
    │           │   └── Item augmentation flow (pipeline)  CifarItem → CifarItem
    │           │       ├── HorizontalFlip (operator)  CifarItem → CifarItem
    │           │       ├── VerticalFlip (operator)  CifarItem → CifarItem
    │           │       ├── AdjustBrightness (operator)  CifarItem → CifarItem
    │           │       └── AdjustContrast (operator)  CifarItem → CifarItem
    │           └── collate (operator)  Iterator[CifarItem] → CifarBatch
    └── Training Pipeline (context_pipeline)  CifarBatch, CifarContext → CifarContext
        ├── train_step (context_operator)  CifarBatch, CifarContext → CifarContext
        ├── logging_step (operator)  CifarContext → CifarContext
        └── save_checkpoint_step (operator)  CifarContext → CifarContext

In [9]:
model = create_cifar10_resnet18()

context = CifarContext(
    model=model,
    optimizer=torch.optim.Adam(model.parameters(), lr=0.01),
    loss_fn=torch.nn.CrossEntropyLoss(),
)


context_epoch1 = train_flow(context)

Iteration 10, Loss: 3.3030


In [10]:
val_split_ratio = 0.02
val_size = int(dataset_size * val_split_ratio)
val_indices = all_indices[-val_size:]

val_source = IcoSource[int](
    fn=lambda: iter(val_indices),
    name="CIFAR10 validation indices",
)
val_input_flow = val_source | IcoOperator(fetch_item).iterate()
val_input_flow.name = "Validation Flow"
val_input_flow.describe()



───────────────────────────────────────────────── Validation Flow ─────────────────────────────────────────────────

Validation Flow (chain)  () → Iterator[CifarItem]
├── CIFAR10 validation indices (source)  () → Iterator[int]
└── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
    └── fetch_item (operator)  int → CifarItem

In [11]:
from dataclasses import dataclass

from apriori.ico.core.context_operator import IcoContextOperator


@dataclass(slots=True)
class CifarEvalContext:
    train_context: CifarContext
    accuracy: float = 0.0
    total_samples: int = 0


def calculate_accuracy(item: CifarItem, context: CifarEvalContext) -> CifarEvalContext:
    with torch.no_grad():
        outputs = context.train_context.model(item.image.unsqueeze(0)) # (1, 10)

        predicted = torch.argmax(outputs, dim=1)
        correct = (predicted.item() == item.label)
        context.total_samples += 1
        context.accuracy += int(correct)

    return context


val_step = IcoContextOperator(calculate_accuracy)
val_epoch = IcoEpoch(
    source=val_input_flow,
    context_operator=val_step,
    name = "Validation Epoch"
)
val_epoch.describe()


──────────────────────────────────────────────── Validation Epoch ─────────────────────────────────────────────────

Validation Epoch (epoch)  Iterator[CifarItem], CifarEvalContext → CifarEvalContext
├── Validation Flow (chain)  () → Iterator[CifarItem]
│   ├── CIFAR10 validation indices (source)  () → Iterator[int]
│   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
│       └── fetch_item (operator)  int → CifarItem
└── calculate_accuracy (context_operator)  CifarItem, CifarEvalContext → CifarEvalContext

In [12]:
def start_eval(context: CifarContext) -> CifarEvalContext:
    context.model.eval()
    return CifarEvalContext(train_context=context)


def log_accuracy(context: CifarEvalContext) -> CifarEvalContext:
    accuracy = context.accuracy / context.total_samples if context.total_samples > 0 else 0.0
    print(f"Validation Accuracy: {accuracy * 100:.2f}% ({context.accuracy}/{context.total_samples})")
    return context


def end_eval(context: CifarEvalContext) -> CifarContext:
    return context.train_context


val_flow = IcoOperator(start_eval) | val_epoch | IcoOperator(log_accuracy) | IcoOperator(end_eval)
val_flow.name = "Validation Flow"
val_flow.describe()

───────────────────────────────────────────────── Validation Flow ─────────────────────────────────────────────────

Validation Flow (chain)  CifarContext → CifarContext
├── start_eval | Validation Epoch | log_accuracy (chain)  CifarContext → CifarEvalContext
│   ├── start_eval | Validation Epoch (chain)  CifarContext → CifarEvalContext
│   │   ├── start_eval (operator)  CifarContext → CifarEvalContext
│   │   └── Validation Epoch (epoch)  Iterator[CifarItem], CifarEvalContext → CifarEvalContext
│   │       ├── Validation Flow (chain)  () → Iterator[CifarItem]
│   │       │   ├── CIFAR10 validation indices (source)  () → Iterator[int]
│   │       │   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
│   │       │       └── fetch_item (operator)  int → CifarItem
│   │       └── calculate_accuracy (context_operator)  CifarItem, CifarEvalContext → CifarEvalContext
│   └── log_accuracy (operator)  CifarEvalContext → CifarEvalContext
└── end_eval (operator)  CifarEvalContext → CifarContext

In [13]:
epoch_flow = train_flow | val_flow
epoch_flow.name = "Train and Validation Epoch Flow"
epoch_flow.describe()

───────────────────────────────────────── Train and Validation Epoch Flow ─────────────────────────────────────────

Train and Validation Epoch Flow (chain)  CifarContext → CifarContext
├── Train Flow (chain)  CifarContext → CifarContext
│   ├── start_train (operator)  CifarContext → CifarContext
│   └── Training Epoch (epoch)  Iterator[CifarBatch], CifarContext → CifarContext
│       ├── Single Process Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
│       │   ├── Train Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
│       │   │   ├── CIFAR10 train indices | shuffle_indices | iterate(fetch_item) (chain)  () → Iterator[CifarItem]
│       │   │   │   ├── CIFAR10 train indices | shuffle_indices (chain)  () → Iterator[int]
│       │   │   │   │   ├── CIFAR10 train indices (source)  () → Iterator[int]
│       │   │   │   │   └── shuffle_indices (operator)  Iterator[int] → Iterator[int]
│       │   │   │   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
│       │   │   │       └── fetch_item (operator)  int → CifarItem
│       │   │   └── batcher(64) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
│       │   └── iterate(Full Augmentation flow) (iterator)  Iterator[Iterator[CifarItem]] → Iterator[CifarBatch]
│       │       └── Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
│       │           ├── iterate(Item augmentation flow) (iterator)  Iterator[CifarItem] → Iterator[CifarItem]
│       │           │   └── Item augmentation flow (pipeline)  CifarItem → CifarItem
│       │           │       ├── HorizontalFlip (operator)  CifarItem → CifarItem
│       │           │       ├── VerticalFlip (operator)  CifarItem → CifarItem
│       │           │       ├── AdjustBrightness (operator)  CifarItem → CifarItem
│       │           │       └── AdjustContrast (operator)  CifarItem → CifarItem
│       │           └── collate (operator)  Iterator[CifarItem] → CifarBatch
│       └── Training Pipeline (context_pipeline)  CifarBatch, CifarContext → CifarContext
│           ├── train_step (context_operator)  CifarBatch, CifarContext → CifarContext
│           ├── logging_step (operator)  CifarContext → CifarContext
│           └── save_checkpoint_step (operator)  CifarContext → CifarContext
└── Validation Flow (chain)  CifarContext → CifarContext
    ├── start_eval | Validation Epoch | log_accuracy (chain)  CifarContext → CifarEvalContext
    │   ├── start_eval | Validation Epoch (chain)  CifarContext → CifarEvalContext
    │   │   ├── start_eval (operator)  CifarContext → CifarEvalContext
    │   │   └── Validation Epoch (epoch)  Iterator[CifarItem], CifarEvalContext → CifarEvalContext
    │   │       ├── Validation Flow (chain)  () → Iterator[CifarItem]
    │   │       │   ├── CIFAR10 validation indices (source)  () → Iterator[int]
    │   │       │   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
    │   │       │       └── fetch_item (operator)  int → CifarItem
    │   │       └── calculate_accuracy (context_operator)  CifarItem, CifarEvalContext → CifarEvalContext
    │   └── log_accuracy (operator)  CifarEvalContext → CifarEvalContext
    └── end_eval (operator)  CifarEvalContext → CifarContext

In [14]:
epoch1_context = epoch_flow(context)


Iteration 20, Loss: 2.5271
Iteration 30, Loss: 2.3059
Validation Accuracy: 12.60% (126.0/1000)


In [15]:
from apriori.ico.core.process import IcoProcess

train_process = IcoProcess(epoch_flow, num_iterations=5)
train_process.name = "CIFAR-10 Training Process"
train_process.describe()


──────────────────────────────────────────── CIFAR-10 Training Process ────────────────────────────────────────────

CIFAR-10 Training Process (process)  CifarContext → CifarContext
└── Train and Validation Epoch Flow (chain)  CifarContext → CifarContext
    ├── Train Flow (chain)  CifarContext → CifarContext
    │   ├── start_train (operator)  CifarContext → CifarContext
    │   └── Training Epoch (epoch)  Iterator[CifarBatch], CifarContext → CifarContext
    │       ├── Single Process Data Flow with Augmentation (chain)  () → Iterator[CifarBatch]
    │       │   ├── Train Input Flow (chain)  () → Iterator[Iterator[CifarItem]]
    │       │   │   ├── CIFAR10 train indices | shuffle_indices | iterate(fetch_item) (chain)  () → 
    │       │   │   │   Iterator[CifarItem]
    │       │   │   │   ├── CIFAR10 train indices | shuffle_indices (chain)  () → Iterator[int]
    │       │   │   │   │   ├── CIFAR10 train indices (source)  () → Iterator[int]
    │       │   │   │   │   └── shuffle_indices (operator)  Iterator[int] → Iterator[int]
    │       │   │   │   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
    │       │   │   │       └── fetch_item (operator)  int → CifarItem
    │       │   │   └── batcher(64) (operator)  Iterator[CifarItem] → Iterator[Iterator[CifarItem]]
    │       │   └── iterate(Full Augmentation flow) (iterator)  Iterator[Iterator[CifarItem]] → 
    │       │       Iterator[CifarBatch]
    │       │       └── Full Augmentation flow (chain)  Iterator[CifarItem] → CifarBatch
    │       │           ├── iterate(Item augmentation flow) (iterator)  Iterator[CifarItem] → Iterator[CifarItem]
    │       │           │   └── Item augmentation flow (pipeline)  CifarItem → CifarItem
    │       │           │       ├── HorizontalFlip (operator)  CifarItem → CifarItem
    │       │           │       ├── VerticalFlip (operator)  CifarItem → CifarItem
    │       │           │       ├── AdjustBrightness (operator)  CifarItem → CifarItem
    │       │           │       └── AdjustContrast (operator)  CifarItem → CifarItem
    │       │           └── collate (operator)  Iterator[CifarItem] → CifarBatch
    │       └── Training Pipeline (context_pipeline)  CifarBatch, CifarContext → CifarContext
    │           ├── train_step (context_operator)  CifarBatch, CifarContext → CifarContext
    │           ├── logging_step (operator)  CifarContext → CifarContext
    │           └── save_checkpoint_step (operator)  CifarContext → CifarContext
    └── Validation Flow (chain)  CifarContext → CifarContext
        ├── start_eval | Validation Epoch | log_accuracy (chain)  CifarContext → CifarEvalContext
        │   ├── start_eval | Validation Epoch (chain)  CifarContext → CifarEvalContext
        │   │   ├── start_eval (operator)  CifarContext → CifarEvalContext
        │   │   └── Validation Epoch (epoch)  Iterator[CifarItem], CifarEvalContext → CifarEvalContext
        │   │       ├── Validation Flow (chain)  () → Iterator[CifarItem]
        │   │       │   ├── CIFAR10 validation indices (source)  () → Iterator[int]
        │   │       │   └── iterate(fetch_item) (iterator)  Iterator[int] → Iterator[CifarItem]
        │   │       │       └── fetch_item (operator)  int → CifarItem
        │   │       └── calculate_accuracy (context_operator)  CifarItem, CifarEvalContext → CifarEvalContext
        │   └── log_accuracy (operator)  CifarEvalContext → CifarEvalContext
        └── end_eval (operator)  CifarEvalContext → CifarContext

In [16]:
final_context = train_process(context)


Iteration 40, Loss: 2.2998
Validation Accuracy: 11.20% (112.0/1000)
Iteration 50, Loss: 2.4220
Iteration 60, Loss: 2.2606
Validation Accuracy: 7.50% (75.0/1000)
Iteration 70, Loss: 2.3430
Iteration 80, Loss: 2.3145
Validation Accuracy: 11.10% (111.0/1000)
Iteration 90, Loss: 2.2928
Validation Accuracy: 9.00% (90.0/1000)
Iteration 100, Loss: 2.2486
Checkpointing model at iteration 100
Iteration 110, Loss: 2.2796
Validation Accuracy: 14.80% (148.0/1000)
